# DPD desde la base de datos (solo lectura)

Lee `scheduled_payments_installments` y `payment_tape` desde **`payments_db`** (MySQL), calcula DPD en ambos modos y exporta a Excel.

**No escribe nada en la base** — solo `SELECT`. Las credenciales salen del `.env` de la raíz del repo (`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`).

El corte por defecto es **el día anterior**. Al correr, el notebook te **pregunta el company_id y el company_code**.

In [ ]:
import os
import sys
from datetime import date

import pandas as pd

sys.path.insert(0, os.path.abspath('.'))
from dpd.integrations.db_excel_runner import (
    run_from_db, prompt_company_id, prompt_company_code, yesterday,
)
from dpd.excel_runner import export_results

# ═══════════════════════════════════════════════════
# PARÁMETROS — editar acá antes de correr
# ═══════════════════════════════════════════════════

CALC_DATE      = None      # None = día anterior (ayer). Podés fijar date(2026, 6, 1).
MODE           = 'both'    # 'cascade' | 'join' | 'both'
GRACE_DAYS     = 1         # días calendario de gracia tras el vencimiento
PARTIAL_COUNTS = False     # solo aplica en cascade
DBNAME         = 'payments_db'   # None = usa DB_NAME del .env
OUT_PATH       = 'resultado_dpd.xlsx'

# Dejá estos en None para que el notebook los pregunte al correr la próxima celda.
COMPANY_ID   = None        # numérico  — filtra payment_tape.company_id
COMPANY_CODE = None        # string    — filtra scheduled_payments_installments.company_code

# ═══════════════════════════════════════════════════

print(f'Corte:      {CALC_DATE or yesterday()} (día anterior)')
print(f'Modo:       {MODE}')
print(f'Grace days: {GRACE_DAYS}')
print(f'Base:       {DBNAME}')

In [ ]:
# === ¿Qué compañía revisamos? ===
# Pregunta company_id y company_code si no los fijaste arriba.
COMPANY_ID   = prompt_company_id(COMPANY_ID)
COMPANY_CODE = prompt_company_code(COMPANY_CODE)
print(f'company_id={COMPANY_ID} | company_code={COMPANY_CODE!r}')

In [ ]:
# === Lectura + cómputo (solo lectura sobre la BD) ===
result_df, summary_df, sched_df, pay_df = run_from_db(
    company_id     = COMPANY_ID,
    company_code   = COMPANY_CODE,
    calc_date      = CALC_DATE,
    mode           = MODE,
    grace_days     = GRACE_DAYS,
    partial_counts = PARTIAL_COUNTS,
    dbname         = DBNAME,
)

In [ ]:
# === Resumen por contrato ===
from IPython.display import display
display(summary_df)

In [ ]:
# === Detalle por cuota ===
with pd.option_context('display.max_rows', 50):
    display(result_df)

In [ ]:
# === Exportar a Excel ===
export_results(result_df, summary_df, OUT_PATH)